## IMPORTS

In [1]:
import pandas as pd
import chromadb
from langchain_openai import OpenAIEmbeddings
from tqdm import tqdm
import os
from dotenv import load_dotenv


load_dotenv("../.env")
load_dotenv("../.secrets")

True

In [2]:
import sys
sys.path.append('../../05_src/')

## DATASET GENERATION

The final data combines soc code and job title with essentail skills, transferable skills, alternate job titles and occupation descriptions 

The soc code and title will make the index and the remaining columns will make the document

Datetime has been added for additional metadata

In [3]:
essential_skills = (
    pd.read_csv("./data/onet/onet_essential_skills.csv")
    .rename(columns = {'element_name' : 'essential_skills'})  
    [["onet_soc_code", "title", "essential_skills"]]  
)
essential_skills = (
    essential_skills
    .groupby(["onet_soc_code", "title"], as_index=False)
    .agg(essential_skills=("essential_skills", lambda s: "\n".join(f"• {item}" for item in s)))
)
essential_skills

,onet_soc_code,title,essential_skills
0,11-1011.00,Chief Executives,• Reading Comprehension\n• Reading Comprehensi...
1,11-1011.03,Chief Sustainability Officers,• Reading Comprehension\n• Reading Comprehensi...
2,11-1021.00,General and Operations Managers,• Reading Comprehension\n• Reading Comprehensi...
3,11-2011.00,Advertising and Promotions Managers,• Reading Comprehension\n• Reading Comprehensi...
4,11-2021.00,Marketing Managers,• Reading Comprehension\n• Reading Comprehensi...
...,...,...,...
905,53-7071.00,Gas Compressor and Gas Pumping Station Operators,• Reading Comprehension\n• Reading Comprehensi...
906,53-7072.00,"Pump Operators, Except Wellhead Pumpers",• Reading Comprehension\n• Reading Comprehensi...
907,53-7073.00,Wellhead Pumpers,• Reading Comprehension\n• Reading Comprehensi...
908,53-7081.00,Refuse and Recyclable Material Collectors,• Reading Comprehension\n• Reading Comprehensi...


In [4]:
transfer_skills = (
    pd.read_csv("./data/onet/onet_transferable_skills.csv")
    .rename(columns = {'element_name' : 'transfer_skills'})  
    [["onet_soc_code", "transfer_skills"]]  
)

transfer_skills = (
    transfer_skills
    .groupby("onet_soc_code", as_index=False)
    .agg(transfer_skills=("transfer_skills", lambda s: "\n".join(f"• {item}" for item in s)))
)
transfer_skills

,onet_soc_code,transfer_skills
0,11-1011.00,• Social Perceptiveness\n• Social Perceptivene...
1,11-1011.03,• Social Perceptiveness\n• Social Perceptivene...
2,11-1021.00,• Social Perceptiveness\n• Social Perceptivene...
3,11-2011.00,• Social Perceptiveness\n• Social Perceptivene...
4,11-2021.00,• Social Perceptiveness\n• Social Perceptivene...
...,...,...
905,53-7071.00,• Social Perceptiveness\n• Social Perceptivene...
906,53-7072.00,• Social Perceptiveness\n• Social Perceptivene...
907,53-7073.00,• Social Perceptiveness\n• Social Perceptivene...
908,53-7081.00,• Social Perceptiveness\n• Social Perceptivene...


In [5]:
job_titles = (
    pd.read_csv("./data/onet/onet_job_titles.csv")
    [["onet_soc_code","job_title"]]  
)

job_titles = (
    job_titles
    .groupby("onet_soc_code", as_index=False)
    .agg(job_title=("job_title", lambda s: "\n".join(f"• {item}" for item in s)))
)

job_titles

,onet_soc_code,job_title
0,11-1011.00,• Aeronautics Commission Director\n• Agency Ow...
1,11-1011.03,• Chief Environmental Commitment Officer (CECO...
2,11-1021.00,• Area Manager\n• Boards and Commissions Direc...
3,11-1031.00,• Alderman\n• Assembly Member\n• Assembly Pers...
4,11-2011.00,• Account Executive\n• Account Manager\n• Ad A...
...,...,...
1011,55-3014.00,• Advanced TOMAHAWK Weapon Control System (ATW...
1012,55-3015.00,• ACDS Block 1 Operator\n• AEGIS Console Opera...
1013,55-3016.00,• Combat Rifle Crewmember\n• Expeditionary For...
1014,55-3018.00,• Advance Seal Delivery System Maintainer\n• A...


In [6]:
desc = (
    pd.read_csv("./data/onet/onet_occupation_data.csv")
    [["onet_soc_code","description"]]  
    )

desc = (
    desc
    .groupby("onet_soc_code", as_index=False)
    .agg(description=("description", lambda s: "\n".join(f"• {item}" for item in s)))
)
desc

,onet_soc_code,description
0,11-1011.00,• Determine and formulate policies and provide...
1,11-1011.03,"• Communicate and coordinate with management, ..."
2,11-1021.00,"• Plan, direct, or coordinate the operations o..."
3,11-1031.00,"• Develop, introduce, or enact laws and statut..."
4,11-2011.00,"• Plan, direct, or coordinate advertising poli..."
...,...,...
1011,55-3014.00,"• Target, fire, and maintain weapons used to d..."
1012,55-3015.00,"• Operate and monitor communications, detectio..."
1013,55-3016.00,• Operate weapons and equipment in ground comb...
1014,55-3018.00,"• Implement unconventional operations by air, ..."


## FINAL MERGE FROM DIFFERENT SOURCES

In [7]:
final = (
    essential_skills
    .merge(
        transfer_skills, on = "onet_soc_code", how = "left"
    )
    .merge(
        job_titles, on = "onet_soc_code", how = "left"
    )
    .merge(
        desc, on = "onet_soc_code", how = "left"
    )
)
final['created_at'] = str(pd.Timestamp.now()) #metadata purposes 
final

,onet_soc_code,title,essential_skills,transfer_skills,job_title,description,created_at
0,11-1011.00,Chief Executives,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Aeronautics Commission Director\n• Agency Ow...,• Determine and formulate policies and provide...,2026-11-16 20:42:16.530734
1,11-1011.03,Chief Sustainability Officers,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Chief Environmental Commitment Officer (CECO...,"• Communicate and coordinate with management, ...",2026-11-16 20:42:16.530734
2,11-1021.00,General and Operations Managers,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Area Manager\n• Boards and Commissions Direc...,"• Plan, direct, or coordinate the operations o...",2026-11-16 20:42:16.530734
3,11-2011.00,Advertising and Promotions Managers,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Account Executive\n• Account Manager\n• Ad A...,"• Plan, direct, or coordinate advertising poli...",2026-11-16 20:42:16.530734
4,11-2021.00,Marketing Managers,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• BD Director (Business Development Director)\...,"• Plan, direct, or coordinate marketing polici...",2026-11-16 20:42:16.530734
...,...,...,...,...,...,...,...
905,53-7071.00,Gas Compressor and Gas Pumping Station Operators,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Acetylene Gas Compressor\n• Air Compressor T...,"• Operate steam-, gas-, electric motor-, or in...",2026-11-16 20:42:16.530734
906,53-7072.00,"Pump Operators, Except Wellhead Pumpers",• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Acid Loader\n• Acid Pump Operator\n• Airveyo...,"• Tend, control, or operate power-driven, stat...",2026-11-16 20:42:16.530734
907,53-7073.00,Wellhead Pumpers,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Field Operator\n• Gas Lease Pumper\n• Gas We...,• Operate power pumps and auxiliary equipment ...,2026-11-16 20:42:16.530734
908,53-7081.00,Refuse and Recyclable Material Collectors,• Reading Comprehension\n• Reading Comprehensi...,• Social Perceptiveness\n• Social Perceptivene...,• Collector\n• Disposal Man\n• Disposal Worker...,• Collect and dump refuse or recyclable materi...,2026-11-16 20:42:16.530734


## ID - METADATA - Document creation

In [8]:
ids = (final['title'] + '-' + final['onet_soc_code']).to_list()

documents = final.apply(
    lambda row: (
        f"Essential Skills:\n{row['essential_skills']}\n"
        f"Transferable Skills:\n{row['transfer_skills']}\n"
        f"Job titles:\n{row['job_title']}"
    ),
    axis=1
).tolist()

metadatas = final[["title", "onet_soc_code", "created_at", "description"]].to_dict("records")

## Save the final document dataset as as JSONL

In [9]:
import json

with open("./documents/soc_job_desc.jsonl", "w", encoding="utf-8") as f:
    for id_, doc, meta in zip(ids, documents, metadatas):
        row = {
            "id": id_,
            "document": doc,
            "metadata": meta
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

## CHUNK THE JSONL FILES

In [10]:
metadatas[0]

{'title': 'Chief Executives',
 'onet_soc_code': '11-1011.00',
 'created_at': '2026-11-16 20:42:16.530734',
 'description': '• Determine and formulate policies and provide overall direction of companies or private and public sector organizations within guidelines set up by a board of directors or similar governing body. Plan, direct, or coordinate operational activities at the highest level of management with the help of subordinate executives and staff managers.'}

In [11]:
from langchain_community.document_loaders import JSONLoader

#Custom metadata function to extract review id from a record
def get_metadata(record:dict, metadata: dict) -> dict:
    record_metadata = record.get('metadata')
    metadata = {}
    metadata['title'] = record_metadata.get('title')
    metadata['onet_soc_code'] = record_metadata.get('onet_soc_code')
    metadata['created_at'] = record_metadata.get('created_at')
    metadata['description'] = record_metadata.get('description')

    return metadata

loader = JSONLoader("./documents/soc_job_desc.jsonl", 
                    jq_schema=".", #Import everything
                    content_key="document",
                    json_lines=True,
                    text_content=True,
                    metadata_func=get_metadata
                    )
data = loader.load()
data[0:10]

/var/folders/9d/_4q678r54f7djh0q_0rsvvhh0000gn/T/ipykernel_26864/1679707057.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


[Document(metadata={'title': 'Chief Executives', 'onet_soc_code': '11-1011.00', 'created_at': '2026-11-16 20:42:16.530734', 'description': '• Determine and formulate policies and provide overall direction of companies or private and public sector organizations within guidelines set up by a board of directors or similar governing body. Plan, direct, or coordinate operational activities at the highest level of management with the help of subordinate executives and staff managers.'}, page_content='Essential Skills:\n• Reading Comprehension\n• Reading Comprehension\n• Active Listening\n• Active Listening\n• Writing\n• Writing\n• Speaking\n• Speaking\n• Mathematics\n• Mathematics\n• Science\n• Science\n• Critical Thinking\n• Critical Thinking\n• Active Learning\n• Active Learning\n• Learning Strategies\n• Learning Strategies\n• Monitoring\n• Monitoring\nTransferable Skills:\n• Social Perceptiveness\n• Social Perceptiveness\n• Coordination\n• Coordination\n• Persuasion\n• Persuasion\n• Negot

In [12]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap=40, #Ensure an intelligent overlap to avoid cutting off sentences
    separators=["."],
    length_function = len,
    add_start_index = True
)
chunks = text_splitter.split_documents(data)
print(f'Split {len(data)} reviews (documents) into {len(chunks)} chunks.' )

Split 910 reviews (documents) into 919 chunks.


In [ ]:
# from IPython.display import Markdown
# for k in range(0, 5):
#     display(Markdown(f"### Chunk {k}"))
#     display(Markdown(f"Metadata: {chunks[k].metadata}"))
#     display(Markdown(chunks[k].page_content))

## Prepare for batching

In [13]:
import json
import os

def prep_batch_file_for_embedding(input:list, output_path:str, max_lines_per_file:int=10):
    total_lines = len(input) #Number of chunks
    num_files = (total_lines // max_lines_per_file) + 1
    os.makedirs(output_path, exist_ok=True)
    print(f'Total lines: {total_lines}, Number of files to create: {num_files}')

    for num_file in range(num_files):
        #going through all the chunks
        start_index = num_file * max_lines_per_file
        end_index = min(start_index + max_lines_per_file, total_lines)
        output_file = os.path.join(output_path, f"job_soc_code_batch_{num_file+1}.jsonl")
        print(f'Creating file: {output_file} with lines from {start_index} to {end_index-1}')
        create_single_batch_file(input, start_index, end_index, output_file)

def create_single_batch_file(input, start_index, end_index, output_file):
    with open(output_file, 'w') as outfile:
        #Number of chunks to process in this batch
        for line in input[start_index:end_index]:
            custom_id = (
                    str(line.metadata['title']) + "_" + 
                    str(line.metadata['onet_soc_code'])
                )
            content = line.page_content
            out_dict = {
                    "custom_id": custom_id, 
                    "method": "POST", 
                    "url": "/v1/embeddings", 
                    "body": {
                        "model": "text-embedding-3-small", 
                        "input": content
                    }
                }
            outfile.write(json.dumps(out_dict) + '\n')

prep_batch_file_for_embedding(
    input=chunks, 
    output_path='./documents/embeddings_batches/'
)

Total lines: 919, Number of files to create: 92
Creating file: ./documents/embeddings_batches/job_soc_code_batch_1.jsonl with lines from 0 to 9
Creating file: ./documents/embeddings_batches/job_soc_code_batch_2.jsonl with lines from 10 to 19
Creating file: ./documents/embeddings_batches/job_soc_code_batch_3.jsonl with lines from 20 to 29
Creating file: ./documents/embeddings_batches/job_soc_code_batch_4.jsonl with lines from 30 to 39
Creating file: ./documents/embeddings_batches/job_soc_code_batch_5.jsonl with lines from 40 to 49
Creating file: ./documents/embeddings_batches/job_soc_code_batch_6.jsonl with lines from 50 to 59
Creating file: ./documents/embeddings_batches/job_soc_code_batch_7.jsonl with lines from 60 to 69
Creating file: ./documents/embeddings_batches/job_soc_code_batch_8.jsonl with lines from 70 to 79
Creating file: ./documents/embeddings_batches/job_soc_code_batch_9.jsonl with lines from 80 to 89
Creating file: ./documents/embeddings_batches/job_soc_code_batch_10.json

## To Open AI

In [14]:
from utils.clients import get_client

client = get_client(use_gateway=True)
files = client.files.list()

In [ ]:
#Used this for deleting the old files 

# for file in files.to_dict()['data']:
#     print(f'Deleting file: {file["filename"]}')
#     resp = client.files.delete(file["id"])
#     print(resp)

In [16]:
from glob import glob

batch_files = glob('./documents/embeddings_batches/job_soc_code_batch_*.jsonl')
batch_files

['./documents/embeddings_batches/job_soc_code_batch_87.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_47.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_63.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_26.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_19.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_61.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_24.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_2.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_45.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_78.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_85.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_41.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_81.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_39.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_58.jsonl',
 './documents/embeddings_batches/job_soc_code_batch_6.js

In [17]:
from tqdm import tqdm

#Store in open AI
my_files = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_files.append(batch_input_file)

100%|██████████| 92/92 [00:49<00:00,  1.87it/s]


In [18]:
my_files[:10]

[FileObject(id='file-DUk2WXdfxZVAnZKWzb8Cru', bytes=45529, created_at=1794879815, filename='job_soc_code_batch_87.jsonl', object='file', purpose='batch', status='processed', expires_at=1797471815, status_details=None),
 FileObject(id='file-KDFLobQXbGMyhuk6QcYeJa', bytes=30410, created_at=1794879816, filename='job_soc_code_batch_47.jsonl', object='file', purpose='batch', status='processed', expires_at=1797471816, status_details=None),
 FileObject(id='file-6J8Zwet3Eaax3fhNP79h2q', bytes=39937, created_at=1794879817, filename='job_soc_code_batch_63.jsonl', object='file', purpose='batch', status='processed', expires_at=1797471817, status_details=None),
 FileObject(id='file-23KmW7zLwYTHafsELVKzzK', bytes=37620, created_at=1794879817, filename='job_soc_code_batch_26.jsonl', object='file', purpose='batch', status='processed', expires_at=1797471817, status_details=None),
 FileObject(id='file-DtrAwhRsuDxzUEnCcZu3cn', bytes=39648, created_at=1794879817, filename='job_soc_code_batch_19.jsonl', ob

## Create the embeddings 

In [19]:
from datetime import datetime

import uuid

my_id = f'deploying-ai-assignment_chat-{uuid.uuid4().hex}'
print(f" my id is : {my_id}")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Job SOC code embeddings ({my_id}) {timestamp}"

for file in tqdm(my_files):
    client.batches.create(
            input_file_id = file.id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

 my id is : deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14


100%|██████████| 92/92 [00:25<00:00,  3.63it/s]


## Store in Chroma DB

In [25]:
# USED TO CHECK STATUS

batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
    'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id'],
    'input_file_id': batch['input_file_id']}
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

[{'batch_id': 'batch_6afbb191e96c81908d9c4413da96d48c',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'in_progress',
  'request_counts': {'completed': 9, 'failed': 0, 'total': 10},
  'output_file_id': None,
  'input_file_id': 'file-3z8jCUXgNKaqTxUW8A5ime'},
 {'batch_id': 'batch_6afbb191b40081909aa366cc691e84a8',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'completed',
  'request_counts': {'completed': 10, 'failed': 0, 'total': 10},
  'output_file_id': 'file-4BLa2pnvnmS5qNhbsRbdLE',
  'input_file_id': 'file-3wRjZqXCaSm8ByW6AjU6tw'},
 {'batch_id': 'batch_6afbb19166f48190ac061622c28af71d',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'in_progress',
  'request_counts': {'completed': 1, 'failed': 0, 'total': 9}

In [26]:
batch_complete = [
        batch  for batch in batch_info if batch['status'] == 'completed'
    ]

batch_complete

[{'batch_id': 'batch_6afbb191b40081909aa366cc691e84a8',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'completed',
  'request_counts': {'completed': 10, 'failed': 0, 'total': 10},
  'output_file_id': 'file-4BLa2pnvnmS5qNhbsRbdLE',
  'input_file_id': 'file-3wRjZqXCaSm8ByW6AjU6tw'},
 {'batch_id': 'batch_6afbb1900e9881909051769d09ee1821',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'completed',
  'request_counts': {'completed': 10, 'failed': 0, 'total': 10},
  'output_file_id': 'file-KiH4X8vakXRxr9zv5f9WQB',
  'input_file_id': 'file-XjHnNJj7VPQ2UTKBCpiZCm'},
 {'batch_id': 'batch_6afbb18fcb508190b498316f14bb2af8',
  'description': 'Job SOC code embeddings (deploying-ai-assignment_chat-ab1c52ad5de94b4cb4bf266e2a942c14) 2026-11-16 20:44:24',
  'status': 'completed',
  'request_counts': {'completed': 10, '

In [33]:
def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines


In [62]:
chroma_inputs = []
#Used a simple approach for chroma inputs
for batch in batch_complete:

    #Get both embedding lines and text lines
    embedding_lines, text_lines = get_text_and_embeddings(batch)

    for embed_item, text_item in zip(embedding_lines, text_lines):
        text = text_item['body']['input']
        custom_id = embed_item["custom_id"]
        chroma_input = {
                    'id': custom_id,
                    'embedding': embed_item['response']['body']['data'][0]['embedding'],
                    'text': text,
                    'metadata':{
                            "custom_id": custom_id,
                            "source": "job_soc_code",
                            "note": "assignment placeholder"
                                }
                        }
        chroma_inputs.append(chroma_input)
  

In [84]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os

CHROMA_PATH = "./chromadb_store"
COLLECTION_NAME = "essential_skills"

EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL') #text-embedding-3-small

#Please run the docker compose here 
CHROMA_URL = os.getenv("CHROMA_URL") 
#The same docker image has been used to access local host here
#This is however not used as persistent client is needed, using CHROMA_PATH now 

def setup_collection(chroma_url:str=CHROMA_URL,
                     collection_name: str = COLLECTION_NAME):

    
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collections = chroma_client.list_collections()

    #Housekeeping
    if collection_name in [col.name for col in collections]:
        chroma_client.delete_collection(name=collection_name)

    #Create a new collection
    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=OpenAIEmbeddingFunction(
            api_key = os.getenv("OPENAI_API_KEY"),
            model_name=EMBEDDING_MODEL)
        )
    return collection

def load_embeddings_to_db(chroma_inputs:list[dict],
                          collection_name:str,
                          chroma_url:str=CHROMA_URL,
                          batch_size:int=1000
                          ):

    #Set up a new collection
    collection = setup_collection(chroma_url=chroma_url, collection_name=collection_name)

    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        collection.add(
            documents=[item['text'] for item in batch],
            embeddings=[item['embedding'] for item in batch],
            metadatas=[item['metadata'] for item in batch],
            ids=[item['id'] for item in batch]
        )

In [85]:
load_embeddings_to_db(chroma_inputs=chroma_inputs,
                      collection_name=COLLECTION_NAME,
                      chroma_url=CHROMA_URL, 
                      batch_size=5)

100%|██████████| 12/12 [00:00<00:00, 66.84it/s]


## Testing the query

In [86]:
collection = client.get_collection(name=COLLECTION_NAME)
print(collection.count())

60


In [90]:
query = "data scientist with Python and SQL"
results = collection.query(
    query_texts=[query],
    n_results=10,
    include=["documents", "metadatas", "distances"]
)

In [92]:
cleaned = []
for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        cleaned.append({
            "title": meta.get("title"),
            "onet_soc_code": meta.get("onet_soc_code"),
            "distance": float(dist),
            "document": doc,
            "metadata": meta
        })


In [93]:
cleaned

[{'title': None,
  'onet_soc_code': None,
  'distance': 0.7136558294296265,
  'document': 'Essential Skills:\n• Reading Comprehension\n• Reading Comprehension\n• Active Listening\n• Active Listening\n• Writing\n• Writing\n• Speaking\n• Speaking\n• Mathematics\n• Mathematics\n• Science\n• Science\n• Critical Thinking\n• Critical Thinking\n• Active Learning\n• Active Learning\n• Learning Strategies\n• Learning Strategies\n• Monitoring\n• Monitoring\nTransferable Skills:\n• Social Perceptiveness\n• Social Perceptiveness\n• Coordination\n• Coordination\n• Persuasion\n• Persuasion\n• Negotiation\n• Negotiation\n• Instructing\n• Instructing\n• Service Orientation\n• Service Orientation\n• Complex Problem Solving\n• Complex Problem Solving\n• Operations Analysis\n• Operations Analysis\n• Technology Design\n• Technology Design\n• Equipment Selection\n• Equipment Selection\n• Installation\n• Installation\n• Programming\n• Programming\n• Operations Monitoring\n• Operations Monitoring\n• Operatio